In [205]:
import pandas as pd
import numpy as np

In [206]:
errors = pd.read_csv('../Data/errors.csv')
failures = pd.read_csv('../Data/failures.csv')
maint = pd.read_csv('../Data/maint.csv')
telemetry = pd.read_csv('../Data/telemetry.csv')
machines = pd.read_csv('../Data/machines.csv')

In [207]:
for df in [errors, failures, maint, telemetry]:
    df['datetime'] = pd.to_datetime(df['datetime'])

In [208]:
for df in [errors, failures, maint, telemetry]:
    df.sort_values(['machineID','datetime'], inplace=True)

In [209]:
error_count = errors.groupby(['machineID','datetime']).size().reset_index(name='error_count')

In [210]:
df = df.merge(error_count, on=['machineID','datetime'], how='left')
df['error_count'].fillna(0, inplace=True)

C:\Users\ShehabYousef\AppData\Local\Temp\ipykernel_29496\4049509397.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['error_count'].fillna(0, inplace=True)


In [211]:
df['error_count'].value_counts()

error_count
0.0    872484
1.0      3342
2.0       245
3.0        29
Name: count, dtype: int64

In [212]:
failures['failure_flag'] = 1

In [213]:
df = df.merge(
    failures[['machineID','datetime','failure_flag']],
    on=['machineID','datetime'],
    how='left'
)

df['failure_flag'].fillna(0, inplace=True)

C:\Users\ShehabYousef\AppData\Local\Temp\ipykernel_29496\1603193733.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['failure_flag'].fillna(0, inplace=True)


In [214]:
df['target'] = df.groupby('machineID')['failure_flag'].shift(-1)
df.dropna(subset=['target'], inplace=True)

In [215]:
maint['maint_flag'] = 1

df = pd.merge_asof(
    df.sort_values('datetime'),
    maint.sort_values('datetime'),
    on='datetime',
    by='machineID',
    direction='backward'
)

In [218]:
df = df.merge(machines, on='machineID', how='left')

In [219]:
df.head()

,datetime,machineID,volt,rotate,pressure,vibration,error_count,failure_flag,target,comp,maint_flag,model,age
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,0.0,0.0,0.0,comp1,1,model3,18
1,2015-01-01 06:00:00,73,167.639992,376.739783,139.337870,38.120657,1.0,0.0,0.0,comp2,1,model2,20
2,2015-01-01 06:00:00,72,167.576902,444.250226,105.874349,41.601923,0.0,0.0,0.0,comp3,1,model4,2
3,2015-01-01 06:00:00,71,171.020312,396.254697,101.927716,40.940543,0.0,0.0,0.0,comp1,1,model2,18
4,2015-01-01 06:00:00,70,162.628154,429.216322,98.680784,43.482879,0.0,0.0,0.0,comp3,1,model3,9
